# Bioassay: Bayesian workflow

**Short Bayesian course — Example 1**

Four dose groups, five animals per group. We model mortality as a function of dose and use the fitted model to estimate the LD50.

Workflow:

$$
\text{data} \rightarrow \text{model} \rightarrow \text{prior predictive}
\rightarrow \text{posterior} \rightarrow \text{LD50}
\rightarrow \text{posterior predictive}
$$


## 0. Setup


In [ ]:
# Uncomment in Colab only if the installed PyMC / ArviZ versions are incompatible.
# %pip install -q "pymc==6.3.2" "arviz==1.3.0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

RANDOM_SEED = 20260921
rng = np.random.default_rng(RANDOM_SEED)

print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)

## 1. Data

At each dose we observe a number of deaths out of five animals.


In [ ]:
bioassay = pd.DataFrame(
    {
        "dose": [-0.86, -0.30, -0.05, 0.73],
        "n": [5, 5, 5, 5],
        "deaths": [0, 1, 3, 5],
    }
)

bioassay["death_rate"] = bioassay["deaths"] / bioassay["n"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(bioassay["dose"], bioassay["death_rate"], s=70)
ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
    title="Bioassay data",
)

plt.show()

## 2. Model

$$
y_j \sim \operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j
$$

with

$$
\alpha\sim\mathcal N(0,5),
\qquad
\beta\sim\operatorname{HalfNormal}(5).
$$

The positive prior on $\beta$ encodes increasing mortality with dose.


In [ ]:
coords = {"dose_group": np.arange(len(bioassay))}

with pm.Model(coords=coords) as model:
    dose = pm.Data(
        "dose",
        bioassay["dose"].to_numpy(),
        dims="dose_group",
    )
    n = pm.Data(
        "n",
        bioassay["n"].to_numpy(),
        dims="dose_group",
    )

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_group",
    )

    deaths = pm.Binomial(
        "deaths",
        n=n,
        p=p,
        observed=bioassay["deaths"].to_numpy(),
        dims="dose_group",
    )

model

## 3. Prior predictive

Before fitting, inspect the dose-response curves implied by the prior.

**Question:** What kinds of relationships have we declared plausible?


In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        samples=1000,
        random_seed=RANDOM_SEED,
    )

alpha_prior = np.asarray(prior["prior"]["alpha"]).reshape(-1)
beta_prior = np.asarray(prior["prior"]["beta"]).reshape(-1)

x_grid = np.linspace(-1.0, 1.0, 200)

draw_ids = rng.choice(len(alpha_prior), size=60, replace=False)
prior_curves = 1 / (
    1 + np.exp(
        -(
            alpha_prior[draw_ids, None]
            + beta_prior[draw_ids, None] * x_grid[None, :]
        )
    )
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(x_grid, prior_curves.T, alpha=0.12, linewidth=1)
ax.scatter(
    bioassay["dose"],
    bioassay["death_rate"],
    s=70,
    zorder=5,
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Probability of death",
    ylim=(-0.05, 1.05),
    title="Curves implied by the prior",
)

plt.show()

## 4. Fit the model

Posterior draws of $(\alpha,\beta)$ represent plausible dose-response relationships after conditioning on the observed data.


In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )

In [ ]:
az.summary(
    idata,
    var_names=["alpha", "beta"],
    hdi_prob=0.90,
    round_to=2,
)

Check $\hat R$, effective sample sizes, and divergences.


In [ ]:
az.plot_trace(idata, var_names=["alpha", "beta"])
plt.show()

## 5. Posterior dose-response curves

Each draw of $(\alpha,\beta)$ defines one curve

$$
p(x)=\operatorname{logistic}(\alpha+\beta x).
$$


In [ ]:
alpha_post = np.asarray(idata["posterior"]["alpha"]).reshape(-1)
beta_post = np.asarray(idata["posterior"]["beta"]).reshape(-1)

posterior_curves = 1 / (
    1 + np.exp(
        -(
            alpha_post[:, None]
            + beta_post[:, None] * x_grid[None, :]
        )
    )
)

curve_median = np.median(posterior_curves, axis=0)
curve_lo, curve_hi = np.quantile(posterior_curves, [0.05, 0.95], axis=0)

fig, ax = plt.subplots(figsize=(7, 4))

ax.fill_between(
    x_grid,
    curve_lo,
    curve_hi,
    alpha=0.25,
    label="Central 90% posterior interval",
)
ax.plot(
    x_grid,
    curve_median,
    linewidth=2,
    label="Posterior median",
)
ax.scatter(
    bioassay["dose"],
    bioassay["death_rate"],
    s=70,
    zorder=5,
    label="Observed proportions",
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Probability of death",
    ylim=(-0.05, 1.05),
    title="Posterior dose-response curve",
)
ax.legend()

plt.show()

## 6. LD50

The LD50 is the dose at which $p=0.5$. Therefore

$$
LD50=-\frac{\alpha}{\beta}.
$$

Compute it for every posterior draw.


In [ ]:
ld50_log_g_ml = -alpha_post / beta_post
ld50_mg_ml = 1000 * np.exp(ld50_log_g_ml)

q05, q50, q95 = np.quantile(ld50_mg_ml, [0.05, 0.50, 0.95])

print(f"Posterior median LD50: {q50:.0f} mg/ml")
print(f"Central 90% credible interval: {q05:.0f} to {q95:.0f} mg/ml")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(ld50_mg_ml, bins=50, density=True, alpha=0.75)
ax.axvline(q50, linestyle="--", label="Posterior median")

ax.set(
    xlabel="LD50 (mg/ml)",
    ylabel="Posterior density",
    title="Posterior distribution of LD50",
)
ax.legend()

plt.show()

**Question:** How is uncertainty about the LD50 different from variability in the outcome of a new group of animals?


## 7. Posterior predictive check

Generate new death counts at the same four doses from the fitted model and compare them with the observed counts.


In [ ]:
with model:
    ppc = pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        random_seed=RANDOM_SEED,
        return_inferencedata=False,
    )

y_rep = np.asarray(ppc["deaths"]).reshape(-1, len(bioassay))

pred_lo, pred_med, pred_hi = np.quantile(
    y_rep,
    [0.05, 0.50, 0.95],
    axis=0,
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    bioassay["dose"],
    pred_med,
    yerr=[pred_med - pred_lo, pred_hi - pred_med],
    fmt="o",
    capsize=5,
    label="Posterior predictive median and central 90%",
)

ax.scatter(
    bioassay["dose"],
    bioassay["deaths"],
    s=80,
    marker="x",
    label="Observed deaths",
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Deaths out of 5",
    ylim=(-0.4, 5.4),
    title="Can the fitted model reproduce data like ours?",
)
ax.legend()

plt.show()

**Question:** Does the fitted model generate datasets that look like the observed one? What kinds of model failures could this tiny dataset reveal—or fail to reveal?


## 8. Workflow recap

$$
\boxed{
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{posterior}
\rightarrow
\text{derived quantity}
\rightarrow
\text{posterior predictive}
}
$$


## 9. Explore

Choose one:

1. Replace the positive prior on $\beta$ with `Normal(0, 5)`. What changes?
2. Narrow the priors and inspect the prior curves before refitting.
3. At a new dose $x=0.2$, compare uncertainty in $p_{\mathrm{new}}$ with uncertainty in the number of deaths among five new animals.
4. Invent a posterior predictive statistic that probes the assumed monotone logistic shape.


## Sources

- Gelman & Vehtari, *Bayesian Workflow*, §3.5: bioassay example.
- Racine-Poon et al. (1986), *Applied Statistics* 35:93–150.
